# 🛍️ AI E-commerce Shopping Agent
A Python starter project for an AI shopping assistant that understands product needs, searches a sample catalog, compares options, and explains recommendations. This notebook uses a local demo catalog; no API key is required for the core workflow.

## 🎯 Project goals
- Parse a natural-language shopping request
- Filter a product catalog by category, budget, and features
- Compare product attributes transparently
- Generate a concise recommendation summary
- Optional extension: connect an LLM or live commerce API

In [ ]:
from dataclasses import dataclass
from typing import List, Optional

In [ ]:
@dataclass
class Product:
    name: str
    category: str
    price_inr: int
    rating: float
    features: List[str]

catalog = [
    Product('Nova X5 Phone', 'smartphone', 21999, 4.4, ['5G', '128GB', '50MP camera', '5000mAh']),
    Product('PixelPro A Phone', 'smartphone', 24999, 4.5, ['5G', '256GB', 'OIS camera', '5000mAh']),
    Product('SoundBeat Headphones', 'audio', 3999, 4.2, ['ANC', 'Bluetooth', '30-hour battery']),
    Product('ViewMax 4K TV', 'television', 42999, 4.3, ['4K', 'Google TV', 'Dolby Audio']),
]
len(catalog)

## 🔎 Product search

In [ ]:
def search_products(category: Optional[str] = None, max_price: Optional[int] = None, required_features: Optional[List[str]] = None):
    required_features = [x.lower() for x in (required_features or [])]
    results = []
    for product in catalog:
        if category and product.category.lower() != category.lower():
            continue
        if max_price is not None and product.price_inr > max_price:
            continue
        feature_text = ' '.join(product.features).lower()
        if any(feature not in feature_text for feature in required_features):
            continue
        results.append(product)
    return sorted(results, key=lambda p: (-p.rating, p.price_inr))

search_products(category='smartphone', max_price=25000, required_features=['5g'])

## ⚖️ Compare matching products

In [ ]:
def compare_products(products):
    if not products:
        return 'No matching products found. Try widening your budget or removing a feature.'
    lines = ['| Product | Price (₹) | Rating | Features |', '|---|---:|---:|---|']
    for p in products:
        lines.append(f'| {p.name} | {p.price_inr:,} | {p.rating}/5 | {", ".join(p.features)} |')
    return '\n'.join(lines)

matches = search_products(category='smartphone', max_price=25000)
print(compare_products(matches))

## 🤖 Explainable recommendation (baseline)

In [ ]:
def recommend(products):
    if not products:
        return 'No products meet the current filters.'
    chosen = products[0]
    return (f'Suggested option: {chosen.name} at ₹{chosen.price_inr:,}. '
            f'It has a {chosen.rating}/5 demo rating and includes {", ".join(chosen.features)}. '
            'Compare current seller price, warranty, return policy, and verified reviews before buying.')

print(recommend(matches))

## 🧠 Optional LLM extension
To turn this baseline into an AI agent, add an LLM call that converts a user's request into structured filters and summarizes retrieved catalog entries. Keep product facts grounded in catalog/API results; do not let the model invent prices, stock, or specifications. Store API credentials in environment variables and never commit `.env`.

## 🗺️ Next steps
- Add CSV/JSON product catalog ingestion
- Add semantic search with embeddings
- Integrate an approved product/search API
- Add price, stock, seller, warranty, and return-policy fields
- Build a Streamlit or React interface
- Add tests, logging, and evaluation for groundedness